# Dashboard: Surveillance (Phase 7)

This dashboard subscribes to live MQTT streams and renders city surveillance state:
- humans movement/state
- illegal events
- camera detections
- active alarms
- inferred camera grid cells (from detection `camera_cell`)

Visual precedence:
1. Base color = deterministic unique per human
2. Human marker from official `criminal_count` (`0`=base, `1-10`=red, `>10`=black)
3. Red alarm blink is a separate marker for one step (`ttl_steps=1`)
4. Camera grid overlay has two layers:
   - light blue = scanned cells (valid `camera_cell` seen)
   - strong blue = detected/covered cells (`detected=true`)

In [1]:
from __future__ import annotations

import colorsys
import json
import math
import threading
import time
from collections import deque
from typing import Any

from IPython.display import display
from anymap_ts import Map
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector

In [2]:
# Config + constants
cfg = load_config()
sim_cfg = cfg.simulation

PARKEN_LAT = 55.7029
PARKEN_LON = 12.5720

SURVEILLANCE_TOPIC_ROOT = "simcity/surveillance"
TOPIC_HUMANS_STATE = f"{SURVEILLANCE_TOPIC_ROOT}/humans/state"
TOPIC_EVENTS_ILLEGAL = f"{SURVEILLANCE_TOPIC_ROOT}/events/illegal"
TOPIC_DETECTIONS = f"{SURVEILLANCE_TOPIC_ROOT}/detections/camera"
TOPIC_ALARMS = f"{SURVEILLANCE_TOPIC_ROOT}/alarms/active"
TOPIC_REGISTRY_UPDATES = f"{SURVEILLANCE_TOPIC_ROOT}/registry/updates"

LIVE_QOS = 0
STEP_DELAY_S = float(sim_cfg.step_delay_s) if (sim_cfg is not None and sim_cfg.step_delay_s > 0) else 1.0

GRID_SIZE = 100
CELL_SIZE_M = 10.0

print(f"Dashboard broker: {cfg.mqtt.host}:{cfg.mqtt.port}")
print(f"Topics -> {TOPIC_HUMANS_STATE}, {TOPIC_EVENTS_ILLEGAL}, {TOPIC_DETECTIONS}, {TOPIC_ALARMS}, {TOPIC_REGISTRY_UPDATES}")
print(f"Dashboard step delay: {STEP_DELAY_S}s")

Dashboard broker: 127.0.0.1:1883
Topics -> simcity/surveillance/humans/state, simcity/surveillance/events/illegal, simcity/surveillance/detections/camera, simcity/surveillance/alarms/active, simcity/surveillance/registry/updates
Dashboard step delay: 1.0s


In [3]:
# Coordinate transform + deterministic colors
def local_xy_to_lnglat(x_m: float, y_m: float, center_lat: float, center_lon: float) -> tuple[float, float]:
    meters_per_deg_lat = 111_320.0
    meters_per_deg_lon = 111_320.0 * math.cos(math.radians(center_lat))
    lat = center_lat + (y_m / meters_per_deg_lat)
    lon = center_lon + (x_m / meters_per_deg_lon)
    return lon, lat

def stable_color_for_human(human_id: str) -> str:
    hue = (sum(ord(ch) for ch in human_id) % 360) / 360.0
    r, g, b = colorsys.hsv_to_rgb(hue, 0.75, 0.95)
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

def normalize_camera_cell(value: Any) -> tuple[int, int] | None:
    if not isinstance(value, (list, tuple)) or len(value) != 2:
        return None
    try:
        cell_x = int(value[0])
        cell_y = int(value[1])
    except Exception:
        return None
    if not (0 <= cell_x < GRID_SIZE and 0 <= cell_y < GRID_SIZE):
        return None
    return cell_x, cell_y

def camera_cell_center_xy(cell: tuple[int, int]) -> tuple[float, float]:
    cell_x, cell_y = cell
    x = (cell_x + 0.5) * CELL_SIZE_M
    y = (cell_y + 0.5) * CELL_SIZE_M
    return x, y

In [4]:
# Map + dashboard state
m = Map(center=(PARKEN_LON, PARKEN_LAT), zoom=15.8, height="700px", width="100%")
m.add_basemap("OpenStreetMap.Mapnik")
display(m)

def create_runtime_state() -> dict[str, Any]:
    return {
        "humans": {},
        "events_by_id": {},
        "latest_detection_by_human": {},
        "active_alarms": {},
        "registry_counts_by_human": {},
        "scanned_human_ids": set(),
        "processed_event_ids": set(),
        "seen_camera_cells": set(),
        "observed_camera_cells": set(),
    }

def create_marker_state() -> dict[str, set[str]]:
    return {
        "known_human_markers": set(),
        "known_alarm_markers": set(),
        "known_camera_seen_markers": set(),
        "known_camera_cell_markers": set(),
    }

def create_stats() -> dict[str, int]:
    return {
        "humans_state": 0,
        "events_illegal": 0,
        "detections": 0,
        "alarms": 0,
        "registry_updates": 0,
        "camera_cells_seen": 0,
        "camera_cells_observed": 0,
        "dropped_invalid": 0,
    }

state_lock = threading.Lock()
inbox: deque[dict[str, Any]] = deque()

runtime_state = create_runtime_state()
humans: dict[str, dict[str, Any]] = runtime_state["humans"]
events_by_id: dict[str, dict[str, Any]] = runtime_state["events_by_id"]
latest_detection_by_human: dict[str, bool] = runtime_state["latest_detection_by_human"]
active_alarms: dict[str, dict[str, Any]] = runtime_state["active_alarms"]
registry_counts_by_human: dict[str, int] = runtime_state["registry_counts_by_human"]
scanned_human_ids: set[str] = runtime_state["scanned_human_ids"]
processed_event_ids: set[str] = runtime_state["processed_event_ids"]
seen_camera_cells: set[tuple[int, int]] = runtime_state["seen_camera_cells"]
observed_camera_cells: set[tuple[int, int]] = runtime_state["observed_camera_cells"]

marker_state = create_marker_state()
known_human_markers: set[str] = marker_state["known_human_markers"]
known_alarm_markers: set[str] = marker_state["known_alarm_markers"]
known_camera_seen_markers: set[str] = marker_state["known_camera_seen_markers"]
known_camera_cell_markers: set[str] = marker_state["known_camera_cell_markers"]

anchor_step: int | None = None
anchor_started_at: float | None = None

stats = create_stats()

# Diagnostic counters for MQTT callback visibility
callback_total = 0
callback_by_topic: dict[str, int] = {}
last_callback_topic: str | None = None

mqtt_connector: MqttConnector | None = None
dashboard_connected = False

def current_local_step_unlocked() -> int | None:
    if anchor_step is None or anchor_started_at is None:
        return None
    elapsed = time.perf_counter() - anchor_started_at
    return anchor_step + int(elapsed // STEP_DELAY_S)

print("Dashboard map initialized.")

Dashboard map initialized.


In [5]:
# MQTT callback ingestion (thread-safe queue only; no UI updates here)
TOPIC_KIND_MAP = {
    TOPIC_HUMANS_STATE: "humans_state",
    TOPIC_EVENTS_ILLEGAL: "events_illegal",
    TOPIC_DETECTIONS: "detections",
    TOPIC_ALARMS: "alarms",
    TOPIC_REGISTRY_UPDATES: "registry_updates",
}

SUBSCRIPTION_TOPICS = (
    TOPIC_HUMANS_STATE,
    TOPIC_EVENTS_ILLEGAL,
    TOPIC_DETECTIONS,
    TOPIC_ALARMS,
    TOPIC_REGISTRY_UPDATES,
)

def set_anchor_from_payload_unlocked(payload: dict[str, Any]) -> None:
    global anchor_step, anchor_started_at
    if anchor_step is not None or "step" not in payload:
        return
    try:
        anchor_step = int(payload["step"])
        anchor_started_at = time.perf_counter()
    except Exception:
        return

def enqueue_message(kind: str, payload: dict[str, Any]) -> None:
    with state_lock:
        set_anchor_from_payload_unlocked(payload)
        inbox.append({"kind": kind, "payload": payload})

def record_callback_topic(topic: str) -> None:
    global callback_total, last_callback_topic
    with state_lock:
        callback_total += 1
        callback_by_topic[topic] = callback_by_topic.get(topic, 0) + 1
        last_callback_topic = topic

def parse_message_payload(message) -> dict[str, Any] | None:
    try:
        return json.loads(message.payload.decode("utf-8"))
    except Exception:
        with state_lock:
            stats["dropped_invalid"] += 1
        return None

def on_any_message(client, userdata, message):
    record_callback_topic(message.topic)

    kind = TOPIC_KIND_MAP.get(message.topic)
    if kind is None:
        return

    payload = parse_message_payload(message)
    if payload is None:
        return

    enqueue_message(kind, payload)

def subscribe_dashboard_topics(client) -> None:
    for topic in SUBSCRIPTION_TOPICS:
        client.subscribe(topic, qos=LIVE_QOS)

try:
    mqtt_connector = MqttConnector(cfg.mqtt, client_id_suffix="dashboard")
    mqtt_connector.connect()

    if mqtt_connector.wait_for_connection(timeout=5.0):
        mqtt_connector.client.on_message = on_any_message
        subscribe_dashboard_topics(mqtt_connector.client)

        dashboard_connected = True
        print("Dashboard subscribed to all surveillance topics.")
    else:
        print("Dashboard MQTT timeout. Running in idle mode.")
except Exception as exc:
    print(f"Dashboard MQTT unavailable ({exc}). Running in idle mode.")

Dashboard subscribed to all surveillance topics.


In [6]:
# Render loop (updates map from queued messages)
def increment_stat(name: str) -> None:
    with state_lock:
        stats[name] += 1

def mark_invalid_message() -> None:
    increment_stat("dropped_invalid")

def handle_humans_state(payload: dict[str, Any]) -> None:
    human_id = str(payload["human_id"])
    humans[human_id] = {
        "name": str(payload.get("name", human_id)),
        "x": float(payload["x"]),
        "y": float(payload["y"]),
        "step": int(payload["step"]),
    }
    increment_stat("humans_state")

def handle_events_illegal(payload: dict[str, Any]) -> None:
    event_id = str(payload["event_id"])
    human_id = str(payload["human_id"])
    events_by_id[event_id] = {
        "step": int(payload["step"]),
        "human_id": human_id,
        "x": float(payload["x"]),
        "y": float(payload["y"]),
    }
    processed_event_ids.add(event_id)
    increment_stat("events_illegal")

def handle_detections(payload: dict[str, Any]) -> None:
    human_id = str(payload["human_id"])
    detected = bool(payload["detected"])
    latest_detection_by_human[human_id] = detected
    scanned_human_ids.add(human_id)

    camera_cell = normalize_camera_cell(payload.get("camera_cell"))
    if camera_cell is not None:
        seen_camera_cells.add(camera_cell)
        stats["camera_cells_seen"] = len(seen_camera_cells)
        if detected:
            observed_camera_cells.add(camera_cell)
            stats["camera_cells_observed"] = len(observed_camera_cells)

    increment_stat("detections")

def handle_alarms(payload: dict[str, Any]) -> None:
    event_id = str(payload["event_id"])
    step = int(payload["step"])
    ttl_steps = int(payload.get("ttl_steps", 1))
    x = float(payload["x"])
    y = float(payload["y"])
    active_alarms[event_id] = {
        "x": x,
        "y": y,
        "until_step": step + ttl_steps,
    }
    increment_stat("alarms")

def handle_registry_updates(payload: dict[str, Any]) -> None:
    human_id = str(payload["human_id"])
    registry_counts_by_human[human_id] = int(payload["criminal_count"])
    increment_stat("registry_updates")

MESSAGE_HANDLERS: dict[str, Any] = {
    "humans_state": handle_humans_state,
    "events_illegal": handle_events_illegal,
    "detections": handle_detections,
    "alarms": handle_alarms,
    "registry_updates": handle_registry_updates,
}

def apply_inbox_messages() -> None:
    while True:
        with state_lock:
            if not inbox:
                break
            item = inbox.popleft()

        kind = item["kind"]
        payload = item["payload"]
        handler = MESSAGE_HANDLERS.get(kind)
        if handler is None:
            continue

        try:
            handler(payload)
        except Exception:
            mark_invalid_message()

def upsert_marker(
    name: str,
    lng: float,
    lat: float,
    color: str,
    popup: str,
    known_markers: set[str],
) -> None:
    if name in known_markers:
        m.remove_marker(name)
    m.add_marker(lng, lat, name=name, color=color, popup=popup)
    known_markers.add(name)

def render_camera_cells(
    cells: set[tuple[int, int]],
    prefix: str,
    color: str,
    popup_key: str,
    known_markers: set[str],
) -> None:
    for cell_x, cell_y in cells:
        marker_name = f"{prefix}:{cell_x}:{cell_y}"
        if marker_name in known_markers:
            continue
        center_x, center_y = camera_cell_center_xy((cell_x, cell_y))
        lng, lat = local_xy_to_lnglat(center_x - 500.0, center_y - 500.0, PARKEN_LAT, PARKEN_LON)
        m.add_marker(
            lng,
            lat,
            name=marker_name,
            color=color,
            popup=f"{popup_key}=[{cell_x}, {cell_y}]",
        )
        known_markers.add(marker_name)

def render_humans() -> None:
    for human_id, state in humans.items():
        base_color = stable_color_for_human(human_id)
        criminal_count = int(registry_counts_by_human.get(human_id, 0))

        if criminal_count > 10:
            marker_color = "#000000"
        elif criminal_count > 0:
            marker_color = "#ff0000"
        else:
            marker_color = base_color

        lng, lat = local_xy_to_lnglat(state["x"] - 500.0, state["y"] - 500.0, PARKEN_LAT, PARKEN_LON)
        upsert_marker(
            name=human_id,
            lng=lng,
            lat=lat,
            color=marker_color,
            popup=f"{state['name']} | criminal_count={criminal_count}",
            known_markers=known_human_markers,
        )

def render_alarms(local_step: int | None) -> None:
    expired_alarm_ids: list[str] = []
    for event_id, alarm in active_alarms.items():
        if local_step is not None and local_step >= int(alarm["until_step"]):
            expired_alarm_ids.append(event_id)
            continue

        alarm_name = f"alarm:{event_id}"
        lng, lat = local_xy_to_lnglat(alarm["x"] - 500.0, alarm["y"] - 500.0, PARKEN_LAT, PARKEN_LON)
        upsert_marker(
            name=alarm_name,
            lng=lng,
            lat=lat,
            color="#ff0000",
            popup=f"ALARM {event_id}",
            known_markers=known_alarm_markers,
        )

    for event_id in expired_alarm_ids:
        alarm_name = f"alarm:{event_id}"
        if alarm_name in known_alarm_markers:
            m.remove_marker(alarm_name)
            known_alarm_markers.remove(alarm_name)
        active_alarms.pop(event_id, None)

def render_dashboard_once() -> None:
    apply_inbox_messages()

    local_step = current_local_step_unlocked()

    render_camera_cells(
        cells=seen_camera_cells,
        prefix="camera-seen",
        color="#9ec5fe",
        popup_key="camera_seen_cell",
        known_markers=known_camera_seen_markers,
    )
    render_camera_cells(
        cells=observed_camera_cells,
        prefix="camera-cell",
        color="#0066ff",
        popup_key="camera_covered_cell",
        known_markers=known_camera_cell_markers,
    )
    render_humans()
    render_alarms(local_step)

def run_dashboard_loop(total_steps: int = 60) -> None:
    print("Dashboard render loop started...")
    for _ in range(total_steps):
        loop_started = time.perf_counter()
        render_dashboard_once()
        elapsed = time.perf_counter() - loop_started
        time.sleep(max(0.0, STEP_DELAY_S - elapsed))
    print("Dashboard render loop complete.")

In [7]:
# Start dashboard loop (run while producer + detector are active)
run_dashboard_loop(total_steps=60)

Dashboard render loop started...
Dashboard render loop complete.


In [8]:
# Investigation: transform sanity + counters + active alarms + scanned humans summary
transform_samples = [
    (0.0, 0.0),
    (500.0, 500.0),
    (1000.0, 1000.0),
    (250.0, 750.0),
]

print("Transform sanity samples (local meters -> lng/lat):")
for x, y in transform_samples:
    lng, lat = local_xy_to_lnglat(x - 500.0, y - 500.0, PARKEN_LAT, PARKEN_LON)
    print(f"- x={x:.1f}, y={y:.1f} -> lng={lng:.6f}, lat={lat:.6f}")

with state_lock:
    local_step_snapshot = current_local_step_unlocked()
    stats_snapshot = dict(stats)
    callback_total_snapshot = callback_total
    callback_by_topic_snapshot = dict(callback_by_topic)
    last_callback_topic_snapshot = last_callback_topic
    inbox_depth_snapshot = len(inbox)

print("\nDashboard counters:")
print(f"- local_current_step={local_step_snapshot}")
for key, value in stats_snapshot.items():
    print(f"- {key}={value}")
print(f"- callback_total={callback_total_snapshot}")
print(f"- last_callback_topic={last_callback_topic_snapshot}")
print(f"- inbox_depth={inbox_depth_snapshot}")

if callback_by_topic_snapshot:
    print("- callback_by_topic:")
    for topic, count in callback_by_topic_snapshot.items():
        print(f"  - {topic}: {count}")

print("\nScanned humans + registry counts:")
if scanned_human_ids:
    for human_id in sorted(scanned_human_ids):
        human_name = humans.get(human_id, {}).get("name", human_id)
        criminal_count = registry_counts_by_human.get(human_id, 0)
        print(f"- {human_name}: criminal_count={criminal_count}")
else:
    print("- none")

print("\nSeen camera cells (light blue layer):")
if seen_camera_cells:
    print(f"- total={len(seen_camera_cells)}")
    sample_cells = sorted(seen_camera_cells)[:10]
    for cell_x, cell_y in sample_cells:
        print(f"  - [{cell_x}, {cell_y}]")
else:
    print("- none")

print("\nObserved camera cells (strong blue layer):")
if observed_camera_cells:
    print(f"- total={len(observed_camera_cells)}")
    sample_cells = sorted(observed_camera_cells)[:10]
    for cell_x, cell_y in sample_cells:
        print(f"  - [{cell_x}, {cell_y}]")
else:
    print("- none")

print("\nActive alarms (event_id -> until_step):")
if active_alarms:
    for event_id, alarm in active_alarms.items():
        print(f"- {event_id}: until_step={alarm['until_step']}")
else:
    print("- none")

Transform sanity samples (local meters -> lng/lat):
- x=0.0, y=0.0 -> lng=12.564029, lat=55.698408
- x=500.0, y=500.0 -> lng=12.572000, lat=55.702900
- x=1000.0, y=1000.0 -> lng=12.579971, lat=55.707392
- x=250.0, y=750.0 -> lng=12.568014, lat=55.705146

Dashboard counters:
- local_current_step=187
- humans_state=600
- events_illegal=142
- detections=140
- alarms=35
- registry_updates=35
- camera_cells_seen=99
- camera_cells_observed=28
- dropped_invalid=0
- callback_total=952
- last_callback_topic=simcity/surveillance/registry/updates
- inbox_depth=0
- callback_by_topic:
  - simcity/surveillance/humans/state: 600
  - simcity/surveillance/events/illegal: 142
  - simcity/surveillance/detections/camera: 140
  - simcity/surveillance/alarms/active: 35
  - simcity/surveillance/registry/updates: 35

Scanned humans + registry counts:
- Ava Jensen: criminal_count=4
- Clara Andersen: criminal_count=3
- Emma Larsen: criminal_count=6
- Freja Rasmussen: criminal_count=7
- Liam Nielsen: criminal_co

In [9]:
# Cleanup
for marker_name in list(known_camera_seen_markers):
    m.remove_marker(marker_name)
known_camera_seen_markers.clear()

for marker_name in list(known_camera_cell_markers):
    m.remove_marker(marker_name)
known_camera_cell_markers.clear()

if mqtt_connector is not None and mqtt_connector.client.is_connected():
    mqtt_connector.disconnect()
    print("Dashboard MQTT connector disconnected.")
else:
    print("Dashboard cleanup: no active MQTT connection.")

Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


Dashboard MQTT connector disconnected.
